# 6 Машины опорных векторов

In [1]:
#!/usr/bin/env python3

import numpy as np
from sklearn import preprocessing
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.svm import SVC, LinearSVC


In [2]:
# Входной файл,
input_file = 'data/income_data.txt'

# Чтение данных
X = []
y = []
count_class1 = 0
count_class2 = 0
max_datapoints = 25000

with open(input_file, 'r') as f:
    for line in f:
        if count_class1 >= max_datapoints and count_class2 >= max_datapoints:
            break
        if '?' in line:
            continue
        data = line[:-1].split(', ')
        if data[-1] == '<=50K' and count_class1 < max_datapoints:
            X.append(data)
            count_class1 += 1
        if data[-1] == '>50K' and count_class2 < max_datapoints:
            X.append(data)
            count_class2 += 1

# Преобразование в массив numpy array
X = np.array(X)

In [3]:
# Преобразование строковых данных в числовые
label_encoder = []
X_encoded = np.empty(X.shape)
for i,item in enumerate(X[0]):
    if item.isdigit():
        X_encoded[:, i] = X[:, i]
    else:
        label_encoder.append(preprocessing.LabelEncoder())
        X_encoded[:, i] = label_encoder[-1].fit_transform(X[:, i])
X = X_encoded[:, :-1].astype(int)
y = X_encoded[:, -1].astype(int)

In [4]:
# Создание SVМ-классификатора
classifier = OneVsOneClassifier(LinearSVC(random_state=0))
# Обучение классификатора
classifier.fit(X, y)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LinearSVC(random_state=0)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes * (n_classes - 1) / 2` OVO problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
classes_ classes_: numpy array of shape [n_classes]Array containing labels.,"ndarray[int64](2,)","[0,1]"
estimators_ estimators_: list of ``n_classes * (n_classes - 1) / 2`` estimatorsEstimators used for predictions.,tuple,"(LinearSVC(random_state=0),)"
n_classes_ n_classes_: intNumber of classes.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,14
"pairwise_indices_ pairwise_indices_: list, length = ``len(estimators_)``, or ``None``Indices of samples used when training the estimators.``None`` when ``estimator``'s `pairwise` tag is False.",NoneType,None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'


In [5]:
# Перекрестная проверка
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)
classifier = OneVsOneClassifier(LinearSVC(random_state=0))
classifier.fit(X_train, y_train)
y_test_pred = classifier.predict(X_test)
# Вычисление F-меры для SVМ-классификатора
f1 = cross_val_score(classifier, X, y, scoring='f1_weighted', cv=3)
print("F1 score: " + str(round(100*f1.mean(), 2)) + "%")

F1 score: 76.08%


In [6]:
# Функция кодирования одной точки данных
def encode_point(input_data, label_encoder):
    encoded = [-1] * len(input_data)
    count = 0
    for i, item in enumerate(input_data):
        if item.isdigit():
            encoded[i] = int(item)
        else:
            encoded[i] = int(label_encoder[count].transform([item])[0])
            count += 1
    return np.array(encoded)

In [7]:
# Предсказание результата для тестовой точки данных
input_data = [
    '37', 'Private', '215646', 'HS-grad', '9', 'Never-married',
    'Handlers-cleaners', 'Not-in-family', 'White', 'Male',
    '0', '0', '40', 'United-States'
]

# Кодирование тестовой точки данных
input_data_encoded = encode_point(input_data, label_encoder)

# Выполнение классификации для кодированной точки данных и вывод результата
predicted_class = classifier.predict([input_data_encoded])
print(label_encoder[-1].inverse_transform(predicted_class)[0])

<=50K


### Эксперименты

In [8]:
# Точки данных для тестирования
test_points = {
    "Точка 1: молодой, HS-grad, 40ч": [
        '37', 'Private', '215646', 'HS-grad', '9', 'Never-married',
        'Handlers-cleaners', 'Not-in-family', 'White', 'Male',
        '0', '0', '40', 'United-States'
    ],
    "Точка 2: зрелый, married, 60ч": [
        '52', 'Self-emp-not-inc', '209642', 'HS-grad', '9', 'Married-civ-spouse',
        'Exec-managerial', 'Husband', 'White', 'Male',
        '0', '0', '60', 'United-States'
    ],
    "Точка 3: молодой, Bachelors, 35ч": [
        '28', 'Private', '336951', 'Bachelors', '13', 'Never-married',
        'Prof-specialty', 'Not-in-family', 'White', 'Female',
        '0', '0', '35', 'United-States'
    ],
    "Точка 4: пограничная, Some-college, 45ч": [
        '45', 'Private', '160178', 'Some-college', '10', 'Divorced',
        'Exec-managerial', 'Unmarried', 'White', 'Female',
        '0', '0', '45', 'United-States'
    ],
}

In [9]:
# Конфигурации классификаторов (ядро + параметры)
classifier_configs = [
    ("SVC linear C=1.0", OneVsOneClassifier(LinearSVC(C=1.0, random_state=0))),
    ("SVC linear C=10.0", OneVsOneClassifier(LinearSVC(C=10.0, random_state=0))),
    ("SVC rbf C=1.0",  OneVsOneClassifier(SVC(kernel='rbf',  C=1.0,  gamma='scale', random_state=0))),
    ("SVC rbf C=10.0", OneVsOneClassifier(SVC(kernel='rbf',  C=10.0, gamma='scale', random_state=0))),
    ("SVC poly deg=2", OneVsOneClassifier(SVC(kernel='poly', degree=2, C=1.0, random_state=0))),
    # ("SVC poly deg=2", OneVsOneClassifier(SVC(kernel='poly', degree=2, C=10.0, random_state=0))),
]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=5
)

for name, clf in classifier_configs:
    print("=" * 70)
    print(f"Классификатор: {name}")
    print("=" * 70)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Метрики на тестовой выборке
    print(f"F1        : {round(100 * f1_score(y_test, y_pred, average='weighted'), 2)}%")
    print(f"Precision : {round(100 * precision_score(y_test, y_pred, average='weighted'), 2)}%")
    print(f"Recall    : {round(100 * recall_score(y_test, y_pred, average='weighted'), 2)}%")

    # Кросс-валидация по F1
    f1_cv = cross_val_score(clf, X, y, scoring='f1_weighted', cv=3)
    print(f"F1 (cv=3) : {round(100 * f1_cv.mean(), 2)}%")

    # Предсказание для каждой точки
    for point_name, point in test_points.items():
        encoded = encode_point(point, label_encoder)
        pred = clf.predict([encoded])[0]
        label = label_encoder[-1].inverse_transform([pred])[0]
        print(f"  {point_name}  ->  {label}")

    print()

Классификатор: SVC linear C=1.0
F1        : 75.75%
Precision : 79.26%
Recall    : 79.56%
F1 (cv=3) : 76.08%
  Точка 1: молодой, HS-grad, 40ч  ->  <=50K
  Точка 2: зрелый, married, 60ч  ->  <=50K
  Точка 3: молодой, Bachelors, 35ч  ->  <=50K
  Точка 4: пограничная, Some-college, 45ч  ->  <=50K

Классификатор: SVC linear C=10.0
F1        : 75.75%
Precision : 79.26%
Recall    : 79.56%
F1 (cv=3) : 76.12%
  Точка 1: молодой, HS-grad, 40ч  ->  <=50K
  Точка 2: зрелый, married, 60ч  ->  <=50K
  Точка 3: молодой, Bachelors, 35ч  ->  <=50K
  Точка 4: пограничная, Some-college, 45ч  ->  <=50K

Классификатор: SVC rbf C=1.0
F1        : 71.51%
Precision : 82.82%
Recall    : 78.19%
F1 (cv=3) : 71.95%
  Точка 1: молодой, HS-grad, 40ч  ->  <=50K
  Точка 2: зрелый, married, 60ч  ->  <=50K
  Точка 3: молодой, Bachelors, 35ч  ->  <=50K
  Точка 4: пограничная, Some-college, 45ч  ->  <=50K

Классификатор: SVC rbf C=10.0
F1        : 73.16%
Precision : 80.36%
Recall    : 78.67%
F1 (cv=3) : 73.74%
  Точка 1: 